<a href="https://colab.research.google.com/github/hiteshraos/BIS-LAB/blob/main/1BM24CS117_GENETIC_ALGORITHM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
import numpy as np

POP_SIZE_IND = 50
POP_SIZE_RES = 50
GENERATIONS = 100
MUTATION_RATE = 0.1

random.seed(42)
np.random.seed(42)

industrial_base = [
    {"id": i, "cost": random.uniform(15, 30), "peak": random.uniform(80, 150)}
    for i in range(POP_SIZE_IND)
]
residential_base = [
    {"id": i, "cost": random.uniform(10, 22), "peak": random.uniform(30, 70)}
    for i in range(POP_SIZE_RES)
]

class Individual:
    def __init__(self, ind_id, res_id):
        self.ind_id = ind_id
        self.res_id = res_id
        self.cost = 0.0
        self.peak = 0.0
        self.rank = 0
        self.crowding_distance = 0.0

    def evaluate(self):
        self.cost = industrial_base[self.ind_id]["cost"] + residential_base[self.res_id]["cost"]
        self.peak = industrial_base[self.ind_id]["peak"] + residential_base[self.res_id]["peak"]

def create_initial_population():
    population = []
    for _ in range(POP_SIZE_IND):
        ind_id = random.randint(0, POP_SIZE_IND - 1)
        res_id = random.randint(0, POP_SIZE_RES - 1)
        ind = Individual(ind_id, res_id)
        ind.evaluate()
        population.append(ind)
    return population

def crossover(parent1, parent2):
    child1_ind = parent1.ind_id if random.random() < 0.5 else parent2.ind_id
    child1_res = parent2.res_id if random.random() < 0.5 else parent1.res_id

    child2_ind = parent2.ind_id if random.random() < 0.5 else parent1.ind_id
    child2_res = parent1.res_id if random.random() < 0.5 else parent2.res_id

    c1 = Individual(child1_ind, child1_res)
    c2 = Individual(child2_ind, child2_res)
    c1.evaluate()
    c2.evaluate()
    return c1, c2

def mutate(individual):
    if random.random() < MUTATION_RATE:
        individual.ind_id = random.randint(0, POP_SIZE_IND - 1)
    if random.random() < MUTATION_RATE:
        individual.res_id = random.randint(0, POP_SIZE_RES - 1)
    individual.evaluate()

def fast_non_dominated_sort(population):
    fronts = [[]]
    for p in population:
        p.domination_count = 0
        p.dominated_solutions = []
        for q in population:
            if (p.cost <= q.cost and p.peak <= q.peak) and (p.cost < q.cost or p.peak < q.peak):
                p.dominated_solutions.append(q)
            elif (q.cost <= p.cost and q.peak <= p.peak) and (q.cost < p.cost or q.peak < p.peak):
                p.domination_count += 1
        if p.domination_count == 0:
            p.rank = 1
            fronts[0].append(p)

    i = 0
    while len(fronts[i]) > 0:
        next_front = []
        for p in fronts[i]:
            for q in p.dominated_solutions:
                q.domination_count -= 1
                if q.domination_count == 0:
                    q.rank = i + 2
                    next_front.append(q)
        i += 1
        fronts.append(next_front)
    return fronts[:-1]

def calculate_crowding_distance(front):
    size = len(front)
    if size == 0:
        return
    for ind in front:
        ind.crowding_distance = 0.0

    front.sort(key=lambda x: x.cost)
    front[0].crowding_distance = float('inf')
    front[-1].crowding_distance = float('inf')
    cost_range = front[-1].cost - front[0].cost if front[-1].cost != front[0].cost else 1
    for i in range(1, size - 1):
        front[i].crowding_distance += (front[i+1].cost - front[i-1].cost) / cost_range

    front.sort(key=lambda x: x.peak)
    front[0].crowding_distance = float('inf')
    front[-1].crowding_distance = float('inf')
    peak_range = front[-1].peak - front[0].peak if front[-1].peak != front[0].peak else 1
    for i in range(1, size - 1):
        front[i].crowding_distance += (front[i+1].peak - front[i-1].peak) / peak_range

population = create_initial_population()

for gen in range(GENERATIONS):
    offspring = []
    while len(offspring) < len(population):
        p1, p2 = random.sample(population, 2)
        c1, c2 = crossover(p1, p2)
        mutate(c1)
        mutate(c2)
        offspring.extend([c1, c2])

    combined_pop = population + offspring[:len(population)]

    fronts = fast_non_dominated_sort(combined_pop)
    new_population = []

    for front in fronts:
        calculate_crowding_distance(front)
        if len(new_population) + len(front) <= POP_SIZE_IND:
            new_population.extend(front)
        else:
            front.sort(key=lambda x: x.crowding_distance, reverse=True)
            new_population.extend(front[:(POP_SIZE_IND - len(new_population))])
            break

    population = new_population

pareto_front = fast_non_dominated_sort(population)[0]
print(f"Optimization complete. Found {len(pareto_front)} optimal balance configurations:\n")
print(f"{'Config #':<10}{'Ind_ID':<8}{'Res_ID':<8}{'Total Cost ($)':<18}{'Total Peak (MW)':<15}")
print("-" * 60)
for idx, ind in enumerate(sorted(pareto_front, key=lambda x: x.cost)):
    print(f"{idx+1:<10}{ind.ind_id:<8}{ind.res_id:<8}{ind.cost:<18.2f}{ind.peak:<15.2f}")


Optimization complete. Found 50 optimal balance configurations:

Config #  Ind_ID  Res_ID  Total Cost ($)    Total Peak (MW)
------------------------------------------------------------
1         47      12      25.96             149.68         
2         47      12      25.96             149.68         
3         44      12      26.37             126.26         
4         44      12      26.37             126.26         
5         44      27      27.82             122.29         
6         44      27      27.82             122.29         
7         44      34      28.93             118.59         
8         44      34      28.93             118.59         
9         44      15      30.05             115.62         
10        44      15      30.05             115.62         
11        33      15      32.13             114.56         
12        33      15      32.13             114.56         
13        4       15      35.02             114.40         
14        4       15      35.02   